# 🛰️ Universal GEE Satellite Scene Finder & Downloader

A reusable Google Colab + Google Earth Engine workflow for discovering satellite acquisitions over a point/buffer or vector AOI, inspecting scene metadata, exporting acquisition reports, and optionally downloading selected scenes.

**Created by:** Shivansh Dutt Shukla ('https://www.linkedin.com/in/shivanshduttshukla/').
## Before you run
- Authenticate with your own Google Earth Engine account.
- If your environment requires a Cloud project, set `GEE_PROJECT` in the first cell to your own project ID.
- Replace the example coordinates, collection ID, dates, and other parameters with your own study requirements before running cell 2.


In [ ]:
!pip install -q openpyxl xlsxwriter

import ee

# Optional: set your Google Cloud / Earth Engine project ID here.
# Leave as None if your Earth Engine Python environment already has a default project.
GEE_PROJECT = None  # Example: "your-ee-project-id"

try:
    if GEE_PROJECT:
        ee.Initialize(project=GEE_PROJECT)
    else:
        ee.Initialize()
    print("✅ Google Earth Engine initialized successfully.")
except Exception as e:
    print(f"Authentication/setup required: {e}")
    ee.Authenticate()
    if GEE_PROJECT:
        ee.Initialize(project=GEE_PROJECT)
    else:
        ee.Initialize()
    print("✅ Google Earth Engine initialized successfully after authentication.")


In [ ]:
import ee
import pandas as pd
import datetime
import os
import re

def _millis_to_datetime(millis):
    """Convert GEE epoch milliseconds to UTC datetime."""
    if millis is None:
        return None
    try:
        return datetime.datetime.fromtimestamp(millis / 1000.0, datetime.timezone.utc)
    except Exception:
        return None

def _safe_get(props, key):
    """Safely extract and format property values."""
    val = props.get(key, None)
    if isinstance(val, list):
        return ", ".join(str(v) for v in val)
    return val

def detect_metadata_keys(sample_props):
    """Dynamically detects cloud, orbit pass, and orbit number keys for ANY collection."""
    all_keys = list(sample_props.keys())

    # Cloud keys priority
    cloud_candidates = [
        'CLOUDY_PIXEL_PERCENTAGE', 'CLOUD_COVER', 'CLOUD_COVER_LAND',
        'cloud_cover', 'cloudScore', 'cloud_coverage', 'CLOUD_PERCENT'
    ]
    detected_cloud_key = None
    for cand in cloud_candidates:
        if cand in all_keys:
            detected_cloud_key = cand
            break

    # Orbit pass candidates
    orbit_pass_candidates = [
        'orbitProperties_pass', 'PassDirection', 'SENSING_ORBIT_DIRECTION',
        'orbit_direction', 'flight_direction', 'orbit_pass', 'PASS'
    ]
    detected_orbit_pass_key = None
    for cand in orbit_pass_candidates:
        if cand in all_keys:
            detected_orbit_pass_key = cand
            break

    # Orbit number candidates
    orbit_num_candidates = [
        'relativeOrbitNumber_start', 'SENSING_ORBIT_NUMBER', 'orbitNumber_start',
        'ORBIT_NUMBER', 'orbit_number', 'relative_orbit', 'RSP_Path_Number',
        'WRS_PATH', 'relativeOrbitNumber_stop'
    ]
    detected_orbit_num_key = None
    for cand in orbit_num_candidates:
        if cand in all_keys:
            detected_orbit_num_key = cand
            break

    return detected_cloud_key, detected_orbit_pass_key, detected_orbit_num_key

def fetch_universal_scene_metadata(
    collection_id: str,
    latitude: float,
    longitude: float,
    fetch_all_acquisitions: bool = True,
    start_date: str = None,
    end_date: str = None,
    cloud_cover_max: float = None,
    orbit_pass: str = 'ALL',
    buffer_meters: int = 1000,
    max_scenes: int = 10000,
):
    """
    Universal metadata extractor for ANY Google Earth Engine ImageCollection.
    Auto-detects cloud cover, orbit parameters, and all available metadata attributes.
    """
    point = ee.Geometry.Point([longitude, latitude])
    roi = point.buffer(buffer_meters)

    try:
        col = ee.ImageCollection(collection_id).filterBounds(roi)
        raw_count = col.size().getInfo()
    except Exception as e:
        print(f"❌ Error loading collection '{collection_id}': {e}")
        return pd.DataFrame(), {}

    if raw_count == 0:
        # Check if collection exists at all or if the point has no coverage
        global_col = ee.ImageCollection(collection_id)
        try:
            global_sample = global_col.first()
            if global_sample is None:
                print(f"⚠️ Collection '{collection_id}' is empty in GEE.")
            else:
                print(f"⚠️ No scenes found over location ({latitude}° N, {longitude}° E) in collection '{collection_id}'.")
        except Exception:
            print(f"❌ Could not access collection '{collection_id}'. Please check if the Collection ID is spelled correctly.")
        return pd.DataFrame(), {}

    # Sample an image to dynamically inspect all available properties
    sample_img = col.first()
    sample_props = sample_img.toDictionary().getInfo()
    cloud_key, orbit_key, orbit_num_key = detect_metadata_keys(sample_props)

    # Determine if radar/optical
    is_radar = (cloud_key is None) or any(term in collection_id.upper() for term in ['SAR', 'S1', 'PALSAR', 'RADAR'])

    print(f"\n📡 Found {raw_count} raw scenes overlapping ({latitude}, {longitude}) in '{collection_id}'")
    print(f"   Sensor type: {'SAR / Radar' if is_radar else 'Optical / Multispectral'}")
    if cloud_key:
        print(f"   Detected Cloud Property: '{cloud_key}'")
    if orbit_key:
        print(f"   Detected Orbit Pass Property: '{orbit_key}'")
    if orbit_num_key:
        print(f"   Detected Orbit Number Property: '{orbit_num_key}'")
    print(f"   Total metadata attributes available: {len(sample_props)}")

    # ── Apply Temporal Filter ──
    if not fetch_all_acquisitions:
        if start_date and end_date:
            col = col.filterDate(start_date, end_date)
            print(f"   Applied Date Filter: {start_date} to {end_date}")
        elif start_date:
            col = col.filterDate(start_date, '2100-01-01')
            print(f"   Applied Date Filter: from {start_date} till date")
        elif end_date:
            col = col.filterDate('1970-01-01', end_date)
            print(f"   Applied Date Filter: up to {end_date}")
    else:
        print("   Date Filter: Complete archive till date (ALL acquisitions)")

    # ── Apply Cloud Cover Filter (Optical only) ──
    if not is_radar and cloud_key and cloud_cover_max is not None and cloud_cover_max < 100:
        col = col.filter(ee.Filter.lte(cloud_key, cloud_cover_max))
        print(f"   Applied Cloud Cover Filter: <= {cloud_cover_max}% (via '{cloud_key}')")

    # ── Apply Orbit Pass Filter ──
    if orbit_pass and orbit_pass.upper() != 'ALL':
        if orbit_key:
            val_upper = orbit_pass.upper()
            val_cap = orbit_pass.capitalize()
            val_lower = orbit_pass.lower()
            col = col.filter(ee.Filter.Or(
                ee.Filter.eq(orbit_key, val_upper),
                ee.Filter.eq(orbit_key, val_cap),
                ee.Filter.eq(orbit_key, val_lower)
            ))
            print(f"   Applied Orbit Pass Filter: '{orbit_pass.upper()}' (via '{orbit_key}')")
        else:
            print(f"   ℹ️ Notice: Orbit pass filtering skipped because '{collection_id}' does not have an orbit pass property.")

    # Sort chronologically
    col = col.sort('system:time_start')
    filtered_count = col.size().getInfo()
    print(f"🎯 Total scenes matching all filter criteria: {filtered_count}")

    if filtered_count == 0:
        print("⚠️ No scenes match your filter criteria. Try relaxing cloud cover or date constraints.")
        return pd.DataFrame(), {}

    if filtered_count > max_scenes:
        print(f"⚠️ Capping results to first {max_scenes} scenes to prevent memory timeout.")
        col = col.limit(max_scenes)

    # ── Collect all property names to extract ──
    base_keys = ['system:index', 'system:time_start', 'system:time_end', 'system:id']
    extract_keys = list(base_keys)
    if cloud_key:
        extract_keys.append(cloud_key)
    if orbit_key:
        extract_keys.append(orbit_key)
    if orbit_num_key:
        extract_keys.append(orbit_num_key)

    for k in sorted(sample_props.keys()):
        if k not in extract_keys and not k.startswith('system:band_'):
            extract_keys.append(k)

    def _extract(image):
        props = {}
        for key in extract_keys:
            props[key] = image.get(key)
        return ee.Feature(None, props)

    print("⏳ Extracting full metadata attributes for all scenes...")
    features = col.map(_extract)
    data = features.getInfo()

    rows = []
    for feat in data['features']:
        p = feat['properties']
        row = {}

        asset_id = _safe_get(p, 'system:id')
        sys_index = _safe_get(p, 'system:index')
        row['Scene_ID'] = sys_index if sys_index else (asset_id.split('/')[-1] if asset_id else 'Unknown')
        row['GEE_Asset_ID'] = asset_id

        dt = _millis_to_datetime(p.get('system:time_start'))
        if dt:
            row['Acquisition_Date'] = dt.strftime('%Y-%m-%d')
            row['Acquisition_Time_UTC'] = dt.strftime('%H:%M:%S')
            row['Day_of_Year'] = dt.timetuple().tm_yday
            row['Day_of_Week'] = dt.strftime('%A')
        else:
            row['Acquisition_Date'] = None
            row['Acquisition_Time_UTC'] = None
            row['Day_of_Year'] = None
            row['Day_of_Week'] = None

        dt_end = _millis_to_datetime(p.get('system:time_end'))
        if dt_end:
            row['Acquisition_End_UTC'] = dt_end.strftime('%Y-%m-%d %H:%M:%S')

        if cloud_key:
            cc = p.get(cloud_key)
            row['Cloud_Cover_%'] = round(cc, 2) if isinstance(cc, (int, float)) else cc

        if orbit_key:
            row['Orbit_Pass'] = _safe_get(p, orbit_key)

        if orbit_num_key:
            row['Orbit_Number'] = _safe_get(p, orbit_num_key)

        # Append all remaining metadata properties
        for key in extract_keys:
            if key not in ['system:index', 'system:id', 'system:time_start', 'system:time_end', cloud_key, orbit_key, orbit_num_key]:
                col_name = key.replace(':', '_')
                row[col_name] = _safe_get(p, key)

        row['Query_Latitude'] = latitude
        row['Query_Longitude'] = longitude
        row['Query_Buffer_M'] = buffer_meters
        rows.append(row)

    df = pd.DataFrame(rows)
    if 'Acquisition_Date' in df.columns:
        df = df.sort_values('Acquisition_Date', ascending=True).reset_index(drop=True)
    df.index += 1
    df.index.name = 'Sr_No'

    metadata_info = {
        'cloud_key': cloud_key,
        'orbit_key': orbit_key,
        'orbit_num_key': orbit_num_key,
        'is_radar': is_radar
    }
    return df, metadata_info

def export_universal_excel(df, filename, collection_id, params_dict):
    """Exports DataFrame to a professional, formatted multi-sheet Excel file."""
    if df.empty:
        print("⚠️ No data to export.")
        return None

    filepath = os.path.join(os.getcwd(), filename)
    with pd.ExcelWriter(filepath, engine='xlsxwriter') as writer:
        workbook = writer.book

        # ── Sheet 1: Scene Metadata ──
        df.to_excel(writer, sheet_name='Scene_Metadata', startrow=2)
        ws1 = writer.sheets['Scene_Metadata']

        title_fmt = workbook.add_format({
            'bold': True, 'font_size': 14, 'font_color': '#1B4F72',
            'bottom': 2, 'bottom_color': '#2E86C1'
        })
        ws1.write(0, 0, f"Scene Acquisition Report — {collection_id}", title_fmt)

        sub_fmt = workbook.add_format({
            'italic': True, 'font_size': 10, 'font_color': '#5D6D7E'
        })
        ws1.write(1, 0, f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S UTC')} | Total Acquisitions: {len(df)}", sub_fmt)

        hdr_fmt = workbook.add_format({
            'bold': True, 'font_color': 'white', 'bg_color': '#2E86C1',
            'border': 1, 'text_wrap': True, 'align': 'center', 'valign': 'vcenter'
        })
        for col_idx, col_name in enumerate(df.columns):
            ws1.write(2, col_idx + 1, col_name, hdr_fmt)
        ws1.write(2, 0, 'Sr_No', hdr_fmt)

        ws1.set_column(0, 0, 8)
        for idx, col in enumerate(df.columns):
            max_len = max(df[col].astype(str).map(len).max() if len(df) > 0 else 0, len(col)) + 3
            max_len = min(max_len, 45)
            ws1.set_column(idx + 1, idx + 1, max_len)

        ws1.freeze_panes(3, 0)

        # ── Sheet 2: Query Parameters ──
        params_df = pd.DataFrame(list(params_dict.items()), columns=['Query_Parameter', 'Value'])
        params_df.to_excel(writer, sheet_name='Query_Parameters', index=False, startrow=1)
        ws2 = writer.sheets['Query_Parameters']
        ws2.write(0, 0, "Query Parameters Audit Log", title_fmt)
        ws2.set_column(0, 0, 28)
        ws2.set_column(1, 1, 55)
        for col_idx, col_name in enumerate(['Query_Parameter', 'Value']):
            ws2.write(1, col_idx, col_name, hdr_fmt)

    print(f"📁 Excel file saved successfully: {filepath}")
    return filepath


In [ ]:
#@title 🛰️ Universal Satellite Scene Acquisition Explorer
#@markdown Run this cell to query any satellite collection in GEE. All inputs are configurable variables below.

# ==============================================================================
# 🛠️ USER CONFIGURATION (CHANGE YOUR VARIABLES HERE)
# ==============================================================================

# 1. Dataset Collection ID: Enter ANY ImageCollection available in GEE
# Popular examples:
#   - 'COPERNICUS/S2_SR_HARMONIZED'          : Sentinel-2 L2A Surface Reflectance (10m)
#   - 'COPERNICUS/S2_HARMONIZED'             : Sentinel-2 L1C TOA Reflectance (10m)
#   - 'COPERNICUS/S1_GRD'                    : Sentinel-1 SAR C-band GRD (10m)
#   - 'LANDSAT/LC09/C02/T1_L2'               : Landsat 9 OLI-2/TIRS-2 Level-2 (30m)
#   - 'LANDSAT/LC08/C02/T1_L2'               : Landsat 8 OLI/TIRS Level-2 (30m)
#   - 'LANDSAT/LE07/C02/T1_L2'               : Landsat 7 ETM+ Level-2 (30m)
#   - 'LANDSAT/LT05/C02/T1_L2'               : Landsat 5 TM Level-2 (30m)
#   - 'JAXA/ALOS/PALSAR-2/Level2_2/ScanSAR'  : ALOS-2 PALSAR-2 L-band ScanSAR (25m)
#   - 'MODIS/061/MOD09GA'                    : MODIS Daily Surface Reflectance (500m)
#   - Or ANY custom GEE ImageCollection ID
COLLECTION_ID = "COPERNICUS/S1_GRD"  #@param ["COPERNICUS/S2_SR_HARMONIZED", "COPERNICUS/S1_GRD", "LANDSAT/LC09/C02/T1_L2", "LANDSAT/LC08/C02/T1_L2", "LANDSAT/LE07/C02/T1_L2", "LANDSAT/LT05/C02/T1_L2", "JAXA/ALOS/PALSAR-2/Level2_2/ScanSAR", "MODIS/061/MOD09GA"] {allow-input: true}

# 2. Geographic Target Location
LATITUDE = 20.5937        #@param {type:"number"}
LONGITUDE = 78.9629       #@param {type:"number"}
BUFFER_METERS = 100    #@param {type:"integer"}

# 3. Temporal Coverage
# Set FETCH_ALL_ACQUISITIONS = True to get all scenes from mission start till date.
# If False, START_DATE and END_DATE below will be used.
FETCH_ALL_ACQUISITIONS = False  #@param {type:"boolean"}
START_DATE = "2025-01-01"      #@param {type:"date"}
END_DATE = "2025-12-31"        #@param {type:"date"}

# 4. Cloud Cover Percentage Filter (Optical Sensors only, automatically bypassed for SAR/Radar)
# Value between 0 and 100. Set to 100 to accept all cloud covers.
CLOUD_COVER_MAX = 5  #@param {type:"slider", min:0, max:100, step:1}

# 5. Orbit Pass Details
# Filter by satellite flight direction: 'ALL', 'ASCENDING', or 'DESCENDING'
ORBIT_PASS = "ALL"  #@param ["ALL", "ASCENDING", "DESCENDING"]

# 6. Output Excel Workbook Options
# Leave empty ("") to auto-generate a descriptive filename like "S2_SR_17.03N_78.18E.xlsx"
OUTPUT_FILENAME = "scene_acquisition_report"   #@param {type:"string"}
AUTO_DOWNLOAD = True   #@param {type:"boolean"}
MAX_SCENES_CAP = 10000 # Safety limit

# ==============================================================================
# 🚀 EXECUTION PIPELINE
# ==============================================================================

print("=" * 80)
print(f"🛰️ UNIVERSAL SCENE ACQUISITION QUERY")
print(f"   Collection ID : {COLLECTION_ID}")
print(f"   Coordinates   : Lat {LATITUDE}° N, Lon {LONGITUDE}° E (Buffer: {BUFFER_METERS} m)")
print(f"   Date Mode     : {'ALL acquisitions till date' if FETCH_ALL_ACQUISITIONS else f'{START_DATE} to {END_DATE}'}")
print(f"   Cloud Cover   : <= {CLOUD_COVER_MAX}% (if optical sensor)")
print(f"   Orbit Pass    : {ORBIT_PASS}")
print("=" * 80)

# Run universal metadata extraction
df, meta_info = fetch_universal_scene_metadata(
    collection_id=COLLECTION_ID,
    latitude=LATITUDE,
    longitude=LONGITUDE,
    fetch_all_acquisitions=FETCH_ALL_ACQUISITIONS,
    start_date=START_DATE,
    end_date=END_DATE,
    cloud_cover_max=CLOUD_COVER_MAX,
    orbit_pass=ORBIT_PASS,
    buffer_meters=BUFFER_METERS,
    max_scenes=MAX_SCENES_CAP
)

if not df.empty:
    print("\n" + "=" * 80)
    print("📊 ACQUISITION SUMMARY & STATISTICS")
    print("=" * 80)
    print(f"Total Scenes Retrieved : {len(df)}")
    if 'Acquisition_Date' in df.columns:
        print(f"Observation Span       : {df['Acquisition_Date'].min()} to {df['Acquisition_Date'].max()}")
        df['Year'] = pd.to_datetime(df['Acquisition_Date']).dt.year
        print("\n📅 Acquisitions per Year:")
        print(df['Year'].value_counts().sort_index().to_string())

    if 'Cloud_Cover_%' in df.columns and df['Cloud_Cover_%'].notna().any():
        print("\n☁️ Cloud Cover Statistics (%):")
        print(f"   Min    : {df['Cloud_Cover_%'].min():.2f}%")
        print(f"   Mean   : {df['Cloud_Cover_%'].mean():.2f}%")
        print(f"   Median : {df['Cloud_Cover_%'].median():.2f}%")
        print(f"   Max    : {df['Cloud_Cover_%'].max():.2f}%")
        print(f"   Clear Scenes (< 10%): {(df['Cloud_Cover_%'] < 10).sum()}")

    if 'Orbit_Pass' in df.columns and df['Orbit_Pass'].notna().any():
        print("\n🔄 Orbit Pass Distribution:")
        print(df['Orbit_Pass'].value_counts().to_string())

    if 'Orbit_Number' in df.columns and df['Orbit_Number'].notna().any():
        print(f"\n🔢 Unique Orbit/Track Numbers: {df['Orbit_Number'].nunique()}")

    # Determine filename
    if not OUTPUT_FILENAME.strip():
        safe_col_name = COLLECTION_ID.split('/')[-1]
        date_tag = "ALL" if FETCH_ALL_ACQUISITIONS else f"{START_DATE}_{END_DATE}"
        OUTPUT_FILENAME = f"{safe_col_name}_{LATITUDE}N_{LONGITUDE}E_{date_tag}.xlsx"
    elif not OUTPUT_FILENAME.endswith('.xlsx'):
        OUTPUT_FILENAME += '.xlsx'

    # Build Audit Log parameters
    audit_params = {
        'Collection ID': COLLECTION_ID,
        'Sensor Type': 'SAR / Radar' if meta_info.get('is_radar') else 'Optical / Multispectral',
        'Query Latitude': LATITUDE,
        'Query Longitude': LONGITUDE,
        'Buffer (meters)': BUFFER_METERS,
        'Temporal Filter': 'All available acquisitions till date' if FETCH_ALL_ACQUISITIONS else f"{START_DATE} to {END_DATE}",
        'Cloud Cover Max Allowed (%)': CLOUD_COVER_MAX if not meta_info.get('is_radar') else 'N/A (Radar)',
        'Orbit Pass Filter': ORBIT_PASS,
        'Detected Cloud Property': meta_info.get('cloud_key') or 'None',
        'Detected Orbit Pass Property': meta_info.get('orbit_key') or 'None',
        'Total Acquisitions Found': len(df),
        'Generated Timestamp (UTC)': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    excel_path = export_universal_excel(df, OUTPUT_FILENAME, COLLECTION_ID, audit_params)

    # Auto download in Colab
    if AUTO_DOWNLOAD and excel_path:
        try:
            from google.colab import files
            files.download(excel_path)
            print(f"\n📥 Browser download initiated in Google Colab for '{OUTPUT_FILENAME}'")
        except Exception as e:
            print(f"Download notice: {e}")

    # Display clean preview table in Colab
    preview_cols = ['Scene_ID', 'Acquisition_Date', 'Acquisition_Time_UTC']
    for extra in ['Orbit_Pass', 'Cloud_Cover_%', 'Orbit_Number', 'Polarizations']:
        if extra in df.columns:
            preview_cols.append(extra)

    # Add first 2 dataset-specific extra columns if available
    for c in df.columns:
        if c not in preview_cols and c not in ['GEE_Asset_ID', 'Day_of_Year', 'Day_of_Week', 'Acquisition_End_UTC', 'Query_Latitude', 'Query_Longitude', 'Query_Buffer_M', 'Year']:
            preview_cols.append(c)
            if len(preview_cols) >= 8:
                break

    print("\n📋 Scene Acquisition Preview (first 25 rows):")
    display(df[preview_cols].head(25))
else:
    print("\n❌ Query returned 0 scenes. Please review coordinates, dates, or collection ID.")


In [ ]:
# CELL 2: Universal Metadata Engine & Scene Downloader Functions

import ee
import os
import requests
import datetime
import pandas as pd
import warnings

def _millis_to_datetime(millis):
    """Convert GEE epoch milliseconds to UTC datetime."""
    if millis is None:
        return None
    try:
        return datetime.datetime.fromtimestamp(millis / 1000.0, datetime.timezone.utc)
    except Exception:
        return None

def _safe_get(props, key):
    """Safely extract and format property values."""
    val = props.get(key, None)
    if isinstance(val, list):
        return ", ".join(str(v) for v in val)
    return val

def detect_metadata_keys(sample_props):
    """Auto-detects cloud, orbit pass, and orbit number property names."""
    all_keys = list(sample_props.keys())

    cloud_candidates = [
        'CLOUDY_PIXEL_PERCENTAGE', 'CLOUD_COVER', 'CLOUD_COVER_LAND',
        'cloud_cover', 'cloudScore', 'cloud_coverage', 'CLOUD_PERCENT'
    ]
    detected_cloud_key = next((k for k in cloud_candidates if k in all_keys), None)

    orbit_pass_candidates = [
        'orbitProperties_pass', 'PassDirection', 'SENSING_ORBIT_DIRECTION',
        'orbit_direction', 'flight_direction', 'orbit_pass', 'PASS'
    ]
    detected_orbit_pass_key = next((k for k in orbit_pass_candidates if k in all_keys), None)

    orbit_num_candidates = [
        'relativeOrbitNumber_start', 'SENSING_ORBIT_NUMBER', 'orbitNumber_start',
        'ORBIT_NUMBER', 'orbit_number', 'relative_orbit', 'RSP_Path_Number',
        'WRS_PATH', 'relativeOrbitNumber_stop'
    ]
    detected_orbit_num_key = next((k for k in orbit_num_candidates if k in all_keys), None)

    return detected_cloud_key, detected_orbit_pass_key, detected_orbit_num_key

def fetch_universal_scene_metadata(
    collection_id: str,
    latitude: float,
    longitude: float,
    fetch_all_acquisitions: bool = True,
    start_date: str = None,
    end_date: str = None,
    cloud_cover_max: float = None,
    orbit_pass: str = 'ALL',
    buffer_meters: int = 1000,
    max_scenes: int = 10000,
):
    """Queries any GEE ImageCollection and extracts all metadata attributes."""
    point = ee.Geometry.Point([longitude, latitude])
    roi = point.buffer(buffer_meters)

    try:
        col = ee.ImageCollection(collection_id).filterBounds(roi)
        raw_count = col.size().getInfo()
    except Exception as e:
        print(f"❌ Error loading collection '{collection_id}': {e}")
        return pd.DataFrame(), {}

    if raw_count == 0:
        print(f"⚠️ No scenes found over location ({latitude}° N, {longitude}° E) in '{collection_id}'.")
        return pd.DataFrame(), {}

    sample_img = col.first()
    sample_props = sample_img.toDictionary().getInfo()
    cloud_key, orbit_key, orbit_num_key = detect_metadata_keys(sample_props)
    is_radar = (cloud_key is None) or any(term in collection_id.upper() for term in ['SAR', 'S1', 'PALSAR', 'RADAR'])

    print(f"\n📡 Found {raw_count} raw scenes overlapping ({latitude}, {longitude}) in '{collection_id}'")
    print(f"   Sensor classification: {'SAR / Radar' if is_radar else 'Optical / Multispectral'}")
    if cloud_key:
        print(f"   Detected Cloud Property: '{cloud_key}'")
    if orbit_key:
        print(f"   Detected Orbit Pass Property: '{orbit_key}'")
    if orbit_num_key:
        print(f"   Detected Orbit Number Property: '{orbit_num_key}'")
    print(f"   Total metadata attributes available: {len(sample_props)}")

    # Temporal filter
    if not fetch_all_acquisitions:
        if start_date and end_date:
            col = col.filterDate(start_date, end_date)
            print(f"   Applied Date Range: {start_date} to {end_date}")
        elif start_date:
            col = col.filterDate(start_date, '2100-01-01')
            print(f"   Applied Date Range: from {start_date} till date")
        elif end_date:
            col = col.filterDate('1970-01-01', end_date)
            print(f"   Applied Date Range: up to {end_date}")
    else:
        print("   Applied Date Range: Complete archive till date (ALL acquisitions)")

    # Cloud cover filter (Optical only)
    if not is_radar and cloud_key and cloud_cover_max is not None and cloud_cover_max < 100:
        col = col.filter(ee.Filter.lte(cloud_key, cloud_cover_max))
        print(f"   Applied Cloud Cover Filter: <= {cloud_cover_max}% (using '{cloud_key}')")

    # Orbit pass filter
    if orbit_pass and orbit_pass.upper() != 'ALL':
        if orbit_key:
            val_upper = orbit_pass.upper()
            val_cap = orbit_pass.capitalize()
            val_lower = orbit_pass.lower()
            col = col.filter(ee.Filter.Or(
                ee.Filter.eq(orbit_key, val_upper),
                ee.Filter.eq(orbit_key, val_cap),
                ee.Filter.eq(orbit_key, val_lower)
            ))
            print(f"   Applied Orbit Pass Filter: '{orbit_pass.upper()}' (using '{orbit_key}')")

    col = col.sort('system:time_start')
    filtered_count = col.size().getInfo()
    print(f"🎯 Total scenes matching all filter criteria: {filtered_count}")

    if filtered_count == 0:
        print("⚠️ No scenes match your filter criteria.")
        return pd.DataFrame(), {}

    if filtered_count > max_scenes:
        print(f"⚠️ Capping results to first {max_scenes} scenes.")
        col = col.limit(max_scenes)

    base_keys = ['system:index', 'system:time_start', 'system:time_end', 'system:id']
    extract_keys = list(base_keys)
    if cloud_key: extract_keys.append(cloud_key)
    if orbit_key: extract_keys.append(orbit_key)
    if orbit_num_key: extract_keys.append(orbit_num_key)

    for k in sorted(sample_props.keys()):
        if k not in extract_keys and not k.startswith('system:band_'):
            extract_keys.append(k)

    def _extract(image):
        props = {}
        for key in extract_keys:
            props[key] = image.get(key)
        return ee.Feature(None, props)

    print("⏳ Extracting full metadata attributes for all scenes...")
    features = col.map(_extract)
    data = features.getInfo()

    rows = []
    for feat in data['features']:
        p = feat['properties']
        row = {}

        asset_id = _safe_get(p, 'system:id')
        sys_index = _safe_get(p, 'system:index')
        row['Scene_ID'] = sys_index if sys_index else (asset_id.split('/')[-1] if asset_id else 'Unknown')
        row['GEE_Asset_ID'] = asset_id

        dt = _millis_to_datetime(p.get('system:time_start'))
        if dt:
            row['Acquisition_Date'] = dt.strftime('%Y-%m-%d')
            row['Acquisition_Time_UTC'] = dt.strftime('%H:%M:%S')
            row['Day_of_Year'] = dt.timetuple().tm_yday
            row['Day_of_Week'] = dt.strftime('%A')
        else:
            row['Acquisition_Date'] = None
            row['Acquisition_Time_UTC'] = None
            row['Day_of_Year'] = None
            row['Day_of_Week'] = None

        dt_end = _millis_to_datetime(p.get('system:time_end'))
        if dt_end:
            row['Acquisition_End_UTC'] = dt_end.strftime('%Y-%m-%d %H:%M:%S')

        if cloud_key:
            cc = p.get(cloud_key)
            row['Cloud_Cover_%'] = round(cc, 2) if isinstance(cc, (int, float)) else cc

        if orbit_key:
            row['Orbit_Pass'] = _safe_get(p, orbit_key)

        if orbit_num_key:
            row['Orbit_Number'] = _safe_get(p, orbit_num_key)

        for key in extract_keys:
            if key not in ['system:index', 'system:id', 'system:time_start', 'system:time_end', cloud_key, orbit_key, orbit_num_key]:
                col_name = key.replace(':', '_')
                row[col_name] = _safe_get(p, key)

        row['Query_Latitude'] = latitude
        row['Query_Longitude'] = longitude
        row['Query_Buffer_M'] = buffer_meters
        rows.append(row)

    df = pd.DataFrame(rows)
    if 'Acquisition_Date' in df.columns:
        df = df.sort_values('Acquisition_Date', ascending=True).reset_index(drop=True)
    df.index += 1
    df.index.name = 'Sr_No'

    metadata_info = {
        'collection_id': collection_id,
        'cloud_key': cloud_key,
        'orbit_key': orbit_key,
        'orbit_num_key': orbit_num_key,
        'is_radar': is_radar,
        'query_lat': latitude,
        'query_lon': longitude,
        'query_buffer': buffer_meters
    }
    return df, metadata_info

def export_universal_excel(df, filename, collection_id, params_dict):
    """Exports DataFrame to formatted Excel file with Metadata and Query Audit Log sheets."""
    if df.empty:
        return None

    filepath = os.path.join(os.getcwd(), filename)
    with pd.ExcelWriter(filepath, engine='xlsxwriter') as writer:
        workbook = writer.book

        df.to_excel(writer, sheet_name='Scene_Metadata', startrow=2)
        ws1 = writer.sheets['Scene_Metadata']

        title_fmt = workbook.add_format({
            'bold': True, 'font_size': 14, 'font_color': '#1B4F72',
            'bottom': 2, 'bottom_color': '#2E86C1'
        })
        ws1.write(0, 0, f"Scene Acquisition Report — {collection_id}", title_fmt)

        sub_fmt = workbook.add_format({'italic': True, 'font_size': 10, 'font_color': '#5D6D7E'})
        ws1.write(1, 0, f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S UTC')} | Total Acquisitions: {len(df)}", sub_fmt)

        hdr_fmt = workbook.add_format({
            'bold': True, 'font_color': 'white', 'bg_color': '#2E86C1',
            'border': 1, 'text_wrap': True, 'align': 'center', 'valign': 'vcenter'
        })
        for col_idx, col_name in enumerate(df.columns):
            ws1.write(2, col_idx + 1, col_name, hdr_fmt)
        ws1.write(2, 0, 'Sr_No', hdr_fmt)

        ws1.set_column(0, 0, 8)
        for idx, col in enumerate(df.columns):
            max_len = max(df[col].astype(str).map(len).max() if len(df) > 0 else 0, len(col)) + 3
            max_len = min(max_len, 45)
            ws1.set_column(idx + 1, idx + 1, max_len)

        ws1.freeze_panes(3, 0)

        params_df = pd.DataFrame(list(params_dict.items()), columns=['Query_Parameter', 'Value'])
        params_df.to_excel(writer, sheet_name='Query_Parameters', index=False, startrow=1)
        ws2 = writer.sheets['Query_Parameters']
        ws2.write(0, 0, "Query Parameters Audit Log", title_fmt)
        ws2.set_column(0, 0, 28)
        ws2.set_column(1, 1, 55)
        for col_idx, col_name in enumerate(['Query_Parameter', 'Value']):
            ws2.write(1, col_idx, col_name, hdr_fmt)

    print(f"📁 Excel file saved: {filepath}")
    return filepath

# ── UNIVERSAL SCENE DOWNLOADER WITH EXTERNAL AOI SUPPORT ─────────────────────

def download_scenes(
    df_or_excel,
    selected_scenes='LATEST',
    bands=None,
    clip_to_roi=True,
    buffer_meters=None,
    scale=None,
    crs='EPSG:4326',
    output_dir='./downloaded_scenes',
    auto_download_colab=True,
    roi_geometry=None,
    aoi_name=None,
    require_point_inside_aoi=True,
):
    """
    Downloads one or multiple scenes from the previously executed scene query.

    If roi_geometry is supplied:
        - Uses the uploaded AOI polygon directly.
        - Checks whether Query_Latitude / Query_Longitude lies inside the AOI.
        - Clips the satellite image to the EXACT uploaded AOI boundary.
        - Downloads the clipped GeoTIFF.

    If roi_geometry is not supplied:
        - Falls back to the original point + buffer method.
    """

    # ======================================================================
    # 1. LOAD THE SCENE METADATA
    # ======================================================================

    if isinstance(df_or_excel, str):

        if not os.path.exists(df_or_excel):
            print(f"❌ Excel file not found: '{df_or_excel}'")
            return []

        print(f"📖 Reading acquisition records from Excel: '{df_or_excel}'")

        df = pd.read_excel(
            df_or_excel,
            sheet_name='Scene_Metadata',
            index_col=0,
            skiprows=2
        )

    elif isinstance(df_or_excel, pd.DataFrame):

        df = df_or_excel.copy()

    else:

        print("❌ Invalid input. Provide either a DataFrame or path to the Excel file.")
        return []

    if df.empty:
        print("⚠️ No scenes available in the dataset to download.")
        return []


    # ======================================================================
    # 2. PARSE SELECTED SCENES
    # ======================================================================

    target_rows = []

    if isinstance(selected_scenes, int):

        target_rows = [selected_scenes]

    elif isinstance(selected_scenes, list):

        if all(isinstance(x, int) for x in selected_scenes):

            target_rows = selected_scenes

        else:

            mask = df['Scene_ID'].isin(selected_scenes)
            target_rows = df[mask].index.tolist()

    elif isinstance(selected_scenes, str):

        sel_upper = selected_scenes.strip().upper()

        if sel_upper == 'LATEST':

            target_rows = [df.index[-1]]

        elif sel_upper == 'FIRST':

            target_rows = [df.index[0]]

        elif sel_upper == 'ALL':

            target_rows = df.index.tolist()

        elif ',' in selected_scenes:

            target_rows = [
                int(x.strip())
                for x in selected_scenes.split(',')
                if x.strip().isdigit()
            ]

        elif selected_scenes.strip().isdigit():

            target_rows = [int(selected_scenes.strip())]

        else:

            mask = df['Scene_ID'] == selected_scenes.strip()
            target_rows = df[mask].index.tolist()


    valid_indices = [
        idx for idx in target_rows
        if idx in df.index
    ]

    if not valid_indices:

        print(
            f"❌ Could not find selected scene(s) "
            f"'{selected_scenes}' in metadata table."
        )

        print(
            f"   Available Sr_No range: "
            f"{df.index.min()} to {df.index.max()}"
        )

        return []


    scenes_to_download = df.loc[valid_indices]

    print("=" * 80)
    print(
        f"📥 PREPARING DOWNLOAD FOR "
        f"{len(scenes_to_download)} SCENE(S)"
    )
    print("=" * 80)


    # ======================================================================
    # 3. PREPARE OUTPUT DIRECTORY
    # ======================================================================

    os.makedirs(output_dir, exist_ok=True)

    downloaded_files = []


    # ======================================================================
    # 4. PROCESS EACH SCENE
    # ======================================================================

    for sr_no, row in scenes_to_download.iterrows():

        asset_id = row.get('GEE_Asset_ID')

        scene_id = row.get(
            'Scene_ID',
            f"scene_{sr_no}"
        )

        date_str = str(
            row.get(
                'Acquisition_Date',
                'unknown_date'
            )
        )

        print()
        print(f"🛰️ Processing [{sr_no}/{len(df)}]: {scene_id}")
        print(
            f"   Date: {date_str} | "
            f"Asset: {asset_id}"
        )


        # ------------------------------------------------------------------
        # Check GEE Asset ID
        # ------------------------------------------------------------------

        if not asset_id or str(asset_id) == 'nan':

            print(
                "   ⚠️ Missing GEE Asset ID, skipping."
            )

            continue


        try:

            # ==============================================================
            # 5. LOAD THE EXACT IMAGE FROM THE PREVIOUS QUERY
            # ==============================================================

            img = ee.Image(asset_id)

            available_bands = img.bandNames().getInfo()


            # ==============================================================
            # 6. BAND SELECTION
            # ==============================================================

            if bands:

                valid_bands = [
                    b for b in bands
                    if b in available_bands
                ]

                if valid_bands:

                    img = img.select(valid_bands)

                    print(
                        f"   Selected bands: {valid_bands}"
                    )

                else:

                    print(
                        f"   ⚠️ Requested bands {bands} "
                        f"not available."
                    )

                    selected = available_bands[
                        :min(4, len(available_bands))
                    ]

                    img = img.select(selected)

                    print(
                        f"   Using fallback bands: {selected}"
                    )

            else:

                # Automatically select useful bands
                if len(available_bands) > 8:

                    common = [
                        b for b in [
                            'B4',
                            'B3',
                            'B2',
                            'B8',
                            'HH',
                            'HV',
                            'VV',
                            'VH'
                        ]
                        if b in available_bands
                    ]

                    selected = (
                        common
                        if common
                        else available_bands[:4]
                    )

                    img = img.select(selected)

                    print(
                        f"   Auto-selected bands "
                        f"({len(selected)}): {selected}"
                    )

                else:

                    print(
                        f"   Bands included "
                        f"({len(available_bands)}): "
                        f"{available_bands}"
                    )


            # ==============================================================
            # 7. DEFINE AOI
            # ==============================================================

            lat = row.get('Query_Latitude')
            lon = row.get('Query_Longitude')


            # ----------------------------------------------------------------
            # OPTION A — UPLOADED SHAPEFILE AOI
            # ----------------------------------------------------------------

            if (
                clip_to_roi
                and roi_geometry is not None
            ):

                print(
                    f"   📐 Using uploaded AOI: "
                    f"{aoi_name or 'Uploaded_AOI'}"
                )


                # ----------------------------------------------------------
                # Check that query coordinate lies inside AOI
                # ----------------------------------------------------------

                if (
                    require_point_inside_aoi
                    and lat is not None
                    and lon is not None
                ):

                    query_point = ee.Geometry.Point(
                        [
                            float(lon),
                            float(lat)
                        ]
                    )

                    point_inside = (
                        roi_geometry
                        .contains(
                            query_point,
                            maxError=1
                        )
                        .getInfo()
                    )

                    if not point_inside:

                        print(
                            "   ❌ Query coordinate is "
                            "OUTSIDE the uploaded AOI."
                        )

                        print(
                            f"      Query point: "
                            f"{lat}, {lon}"
                        )

                        print(
                            "      Scene will NOT be downloaded."
                        )

                        continue

                    print(
                        "   ✅ Query coordinate lies "
                        "inside uploaded AOI."
                    )


                # ----------------------------------------------------------
                # Clip using EXACT uploaded polygon
                # ----------------------------------------------------------

                img = img.clip(roi_geometry)

                export_region = roi_geometry

                region_desc = (
                    aoi_name
                    if aoi_name
                    else "Uploaded_AOI"
                )


            # ----------------------------------------------------------------
            # OPTION B — ORIGINAL POINT + BUFFER METHOD
            # ----------------------------------------------------------------

            elif (
                clip_to_roi
                and lat is not None
                and lon is not None
            ):

                buf = (
                    buffer_meters
                    if buffer_meters is not None
                    else row.get(
                        'Query_Buffer_M',
                        1000
                    )
                )

                roi_geom = (
                    ee.Geometry.Point(
                        [
                            float(lon),
                            float(lat)
                        ]
                    )
                    .buffer(float(buf))
                    .bounds()
                )

                img = img.clip(roi_geom)

                export_region = roi_geom

                region_desc = (
                    f"AOI_{int(buf)}m"
                )


            # ----------------------------------------------------------------
            # OPTION C — FULL SCENE
            # ----------------------------------------------------------------

            else:

                export_region = img.geometry()

                region_desc = "FullScene"


            # ==============================================================
            # 8. AUTO-DETECT PIXEL SIZE
            # ==============================================================

            if scale is None:

                try:

                    nominal_scale = (
                        img
                        .select(0)
                        .projection()
                        .nominalScale()
                        .getInfo()
                    )

                    current_scale = max(
                        float(nominal_scale),
                        10.0
                    )

                except Exception:

                    current_scale = 30.0

            else:

                current_scale = float(scale)


            print(
                f"   Resolution scale: "
                f"{current_scale} m"
            )

            print(
                f"   Download region: "
                f"{region_desc}"
            )


            # ==============================================================
            # 9. CREATE SAFE OUTPUT FILENAME
            # ==============================================================

            safe_scene_name = "".join(
                c
                for c in str(scene_id)
                if c.isalnum()
                or c in ('-', '_')
            ).rstrip()


            safe_aoi_name = "".join(
                c
                for c in str(region_desc)
                if c.isalnum()
                or c in ('-', '_')
            ).rstrip()


            out_filename = (
                f"{safe_scene_name}_"
                f"{date_str}_"
                f"{safe_aoi_name}.tif"
            )

            out_filepath = os.path.join(
                output_dir,
                out_filename
            )


            # ==============================================================
            # 10. DIRECT EARTH ENGINE DOWNLOAD
            # ==============================================================

            try:

                print(
                    "   ⏳ Requesting direct "
                    "GeoTIFF download URL..."
                )

                download_params = {
                    'scale': current_scale,
                    'crs': crs,
                    'region': export_region,
                    'format': 'GEO_TIFF'
                }


                url = img.getDownloadURL(
                    download_params
                )


                print(
                    "   ⬇️ Streaming GeoTIFF "
                    "to system..."
                )


                response = requests.get(
                    url,
                    stream=True,
                    timeout=300
                )


                if response.status_code == 200:

                    total_bytes = 0

                    with open(
                        out_filepath,
                        'wb'
                    ) as f:

                        for chunk in response.iter_content(
                            chunk_size=65536
                        ):

                            if chunk:

                                f.write(chunk)

                                total_bytes += len(chunk)


                    size_mb = (
                        total_bytes /
                        (1024 * 1024)
                    )


                    print(
                        f"   ✅ Saved: "
                        f"{out_filepath} "
                        f"({size_mb:.2f} MB)"
                    )


                    downloaded_files.append(
                        out_filepath
                    )


                    # ------------------------------------------------------
                    # Trigger browser download
                    # ------------------------------------------------------

                    if auto_download_colab:

                        try:

                            from google.colab import files

                            files.download(
                                out_filepath
                            )

                            print(
                                f"   📥 Browser download "
                                f"triggered for "
                                f"'{out_filename}'"
                            )

                        except Exception as e:

                            print(
                                f"   ℹ️ File saved locally "
                                f"but browser download "
                                f"could not be triggered: "
                                f"{e}"
                            )


                else:

                    print(
                        f"   ❌ HTTP Error "
                        f"{response.status_code}"
                    )

                    print(
                        response.text[:500]
                    )


            # ==============================================================
            # 11. FALLBACK — GOOGLE DRIVE EXPORT
            # ==============================================================

            except Exception as dl_err:

                print(
                    f"   ⚠️ Direct download failed: "
                    f"{dl_err}"
                )

                print(
                    "   🔄 Starting Google Drive "
                    "export as fallback..."
                )


                task = ee.batch.Export.image.toDrive(
                    image=img,
                    description=safe_scene_name[:100],
                    folder='GEE_Downloads',
                    fileNamePrefix=(
                        f"{safe_scene_name}_"
                        f"{safe_aoi_name}"
                    ),
                    region=export_region,
                    scale=current_scale,
                    crs=crs,
                    maxPixels=1e13
                )

                task.start()

                task_id = task.id

                print(
                    f"   🚀 Export task started."
                )

                print(
                    f"      Task ID: {task_id}"
                )

                downloaded_files.append(
                    f"Drive_Task_{task_id}"
                )


        except Exception as err:

            print(
                f"   ❌ Failed to process "
                f"scene {scene_id}: {err}"
            )


    # ======================================================================
    # 12. FINAL SUMMARY
    # ======================================================================

    print()
    print("=" * 80)

    print(
        f"🎉 DOWNLOAD PROCESS COMPLETED: "
        f"{len(downloaded_files)} item(s) processed."
    )

    print("=" * 80)

    return downloaded_files


**Option to Download one or multiple scene Clipped to the Define Buffer Area **

In [ ]:
# CELL 3: Download Scene(s) Directly to System
# You can pass the in-memory `df` OR the path to the exported Excel sheet!

# ==============================================================================
# 🛠️ SCENE DOWNLOAD CONFIGURATION VARIABLES
# ==============================================================================

# 1. Which scenes to download?
# Examples:
#   - 'LATEST'      : Download the most recent scene
#   - 'FIRST'       : Download the oldest scene
#   - 1             : Download scene with Sr_No 1
#   - [1, 5, 10]    : List of specific Sr_No numbers
#   - '1, 3, 5'     : Comma-separated string of Sr_No numbers
#   - 'ALL'         : Download all scenes found
#   - 'ALOS2...'    : Specific Scene_ID string
SCENES_TO_DOWNLOAD = "1"  #@param {type:"string"}

# 2. Area of Interest (AOI) Clipping Option
# If True (Recommended), clips the download to your target coordinates + buffer in meters.
# Direct download as GeoTIFF completes in seconds.
# If False, exports the entire full satellite scene footprint.
CLIP_TO_AOI = True  #@param {type:"boolean"}
BUFFER_METERS = 1000  #@param {type:"integer"}

# 3. Band Selection
# Specify bands as a list (e.g. ['B4', 'B3', 'B2'] for RGB, ['HH', 'HV'] for PALSAR-2).
# Set to None to download standard bands automatically.
BANDS = None

# 4. Input Source
# Pass path to the Excel file OR in-memory DataFrame:
EXCEL_FILE = ""

# 5. Output directory & Colab download
OUTPUT_DIRECTORY = "./downloaded_scenes"
AUTO_DOWNLOAD_TO_PC = False

# Execute Download
downloaded_files = download_scenes(
    df_or_excel=EXCEL_FILE if os.path.exists(EXCEL_FILE) else df,
    selected_scenes=SCENES_TO_DOWNLOAD,
    bands=BANDS,
    clip_to_roi=CLIP_TO_AOI,
    buffer_meters=BUFFER_METERS,
    output_dir=OUTPUT_DIRECTORY,
    auto_download_colab=AUTO_DOWNLOAD_TO_PC
)

In [ ]:
#@title 🛰️ Interactive Shapefile / Vector AOI Scene Downloader
#@markdown Set the download options below (same style as the "Universal Satellite
#@markdown Scene Acquisition Explorer" cell), then run this cell. Upload, scene and
#@markdown band pickers appear as an interactive panel underneath, since those
#@markdown depend on the file/scene you pick at runtime.
#@markdown
#@markdown Supports: **ZIP Shapefile**, loose **.shp** (+ .shx/.dbf/.prj), **GeoJSON**,
#@markdown **GeoPackage (.gpkg)** — including multi-layer — and **KML**.

# ==============================================================================
# 🛠️ DOWNLOAD SETTINGS (CHANGE YOUR VARIABLES HERE)
# ==============================================================================

# 1. AOI Clipping
CLIP_TO_SHAPEFILE = True             #@param {type:"boolean"}
REQUIRE_POINT_INSIDE_AOI = True      #@param {type:"boolean"}

# 2. Export Resolution
EXPORT_SCALE_METERS = 10             #@param {type:"integer"}

# 3. Output
OUTPUT_FILENAME = "AOI_Scene"                       #@param {type:"string"}
OUTPUT_DIRECTORY = "/content/satellite_downloads"   #@param {type:"string"}
AUTO_DOWNLOAD = True                                #@param {type:"boolean"}

# 4. Large-Scene Fallback
# If the selected scene/AOI is too large for a direct download (Earth Engine
# enforces a request-size limit on getDownloadURL), automatically submit a GEE
# batch Export task that writes the GeoTIFF to your Google Drive instead.
ENABLE_DRIVE_FALLBACK = True                #@param {type:"boolean"}
DRIVE_EXPORT_FOLDER = "GEE_Downloads"       #@param {type:"string"}

# ==============================================================================
# 🚀 EXECUTION — sets up the interactive panel below
# ==============================================================================

import os
import zipfile
import requests
import pandas as pd
import geopandas as gpd
import ee

import ipywidgets as widgets
from IPython.display import display, clear_output

os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)
WORKDIR = "/content/uploaded_aoi"
os.makedirs(WORKDIR, exist_ok=True)


# ============================================================
# CHECK QUERY DATAFRAME
# ============================================================

if "df" not in globals():
    raise RuntimeError(
        "❌ 'df' was not found.\n"
        "Please run your satellite acquisition/query cell first."
    )

if df is None or len(df) == 0:
    raise RuntimeError(
        "❌ The query dataframe 'df' is empty.\n"
        "Please run a query that returns at least one scene."
    )

required_columns = ["GEE_Asset_ID"]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise RuntimeError(f"❌ Required column(s) missing from df: {missing_columns}")


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_scene_label(row, index):
    """Create a readable scene name for the dropdown."""
    scene_id = str(row["GEE_Asset_ID"])
    date_text = ""
    for col in ["Acquisition_Date", "Date", "DATE", "system:time_start"]:
        if col in row.index:
            value = row[col]
            if pd.notna(value):
                date_text = str(value)
                break
    if date_text:
        return f"{index} | {date_text} | {scene_id}"
    return f"{index} | {scene_id}"


def get_query_coordinates(row):
    """Find query latitude/longitude from dataframe row."""
    lat_candidates = ["Query_Latitude", "Latitude", "LATITUDE", "lat", "Lat"]
    lon_candidates = ["Query_Longitude", "Longitude", "LONGITUDE", "lon", "Lon"]
    lat = lon = None
    for c in lat_candidates:
        if c in row.index and pd.notna(row[c]):
            try:
                lat = float(row[c]); break
            except Exception:
                pass
    for c in lon_candidates:
        if c in row.index and pd.notna(row[c]):
            try:
                lon = float(row[c]); break
            except Exception:
                pass
    return lat, lon


def get_available_bands(asset_id):
    """Get the actual bands available in the selected Earth Engine image."""
    img = ee.Image(asset_id)
    bands = img.bandNames().getInfo()
    return img, bands


def _safe_name(s):
    cleaned = "".join(c for c in str(s) if c.isalnum() or c in ("-", "_")).rstrip()
    return cleaned or "AOI_Scene"


def _find_files(root, exts):
    hits = []
    for r, _dirs, files_list in os.walk(root):
        for f in files_list:
            if f.lower().endswith(exts):
                hits.append(os.path.join(r, f))
    return hits


def _extract_uploaded_items(value):
    """
    Normalizes ipywidgets FileUpload.value across API versions:
      - ipywidgets 7.x : dict  {filename: {'metadata':..., 'content': bytes}}
      - ipywidgets 8.x : tuple of dict-like items with 'name'/'content' keys
    Returns a list of (filename, bytes) tuples.
    """
    items = []
    if isinstance(value, dict):
        for fname, info in value.items():
            content = info.get("content")
            items.append((fname, bytes(content)))
    else:
        for info in value:
            if isinstance(info, dict):
                fname = info.get("name")
                content = info.get("content")
            else:
                fname = getattr(info, "name", None)
                content = getattr(info, "content", None)
            items.append((fname, bytes(content)))
    return items


def _resolve_main_vector_path(paths):
    """Pick the 'main' AOI file among possibly several uploaded files."""
    priority = (".zip", ".gpkg", ".geojson", ".json", ".kml", ".shp")
    for ext in priority:
        for p in paths:
            if p.lower().endswith(ext):
                return p
    return None


def _extract_zip_and_find_vector(path):
    extract_dir = os.path.join(WORKDIR, "extracted_" + os.path.splitext(os.path.basename(path))[0])
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(path, "r") as z:
        z.extractall(extract_dir)

    shp_files = _find_files(extract_dir, (".shp",))
    gpkg_files = _find_files(extract_dir, (".gpkg",))
    geojson_files = _find_files(extract_dir, (".geojson", ".json"))
    kml_files = _find_files(extract_dir, (".kml",))

    if shp_files:
        return shp_files[0], "shp"
    if gpkg_files:
        return gpkg_files[0], "gpkg"
    if geojson_files:
        return geojson_files[0], "geojson"
    if kml_files:
        return kml_files[0], "kml"
    raise ValueError("❌ No .shp, .gpkg, .geojson, or .kml found inside the ZIP.")


def _list_layers(path):
    """List layer names of a GeoPackage. Returns [] if not applicable."""
    try:
        return gpd.list_layers(path)["name"].tolist()
    except Exception:
        try:
            import fiona
            return fiona.listlayers(path)
        except Exception:
            return []


def _read_kml(path):
    try:
        return gpd.read_file(path, driver="KML")
    except Exception:
        try:
            import fiona
            fiona.drvsupport.supported_drivers["KML"] = "rw"
            fiona.drvsupport.supported_drivers["LIBKML"] = "rw"
        except Exception:
            pass
        return gpd.read_file(path, driver="KML")


def load_vector_aoi(path, kind, layer_name=None):
    """
    Loads an AOI from a resolved vector file path, reprojects to EPSG:4326,
    and returns (gdf, ee.Geometry).
    """
    if kind == "shp":
        gdf = gpd.read_file(path)
    elif kind == "gpkg":
        gdf = gpd.read_file(path, layer=layer_name) if layer_name else gpd.read_file(path)
    elif kind == "geojson":
        gdf = gpd.read_file(path)
    elif kind == "kml":
        gdf = _read_kml(path)
    else:
        raise ValueError(f"❌ Unsupported AOI type: {kind}")

    if gdf.empty:
        raise ValueError("❌ Uploaded AOI contains no features.")

    if gdf.crs is None:
        print("   ⚠️ No CRS found in AOI file — assuming EPSG:4326 (lat/lon).")
        gdf = gdf.set_crs("EPSG:4326")
    else:
        gdf = gdf.to_crs("EPSG:4326")

    gdf = gdf[gdf.geometry.notnull() & (~gdf.geometry.is_empty)]
    if gdf.empty:
        raise ValueError("❌ No valid geometries found after cleaning.")

    try:
        aoi_geometry = gdf.geometry.union_all()
    except AttributeError:
        aoi_geometry = gdf.geometry.unary_union

    ee_aoi = ee.Geometry(aoi_geometry.__geo_interface__)
    return gdf, ee_aoi


# ============================================================
# CREATE INTERACTIVE WIDGETS (upload / scene / bands only —
# the settings above already came from the form fields)
# ============================================================

title = widgets.HTML(
    value="""
    <h2>🛰️ Shapefile / Vector AOI Scene Downloader</h2>
    <p>
    Upload an AOI file and select a scene + bands below.
    Clip / scale / output / Drive-fallback options are set in the
    fields at the top of this cell.
    </p>
    """
)

# ------------------------------------------------------------
# AOI UPLOAD
# ------------------------------------------------------------

upload_widget = widgets.FileUpload(
    accept=".zip,.shp,.shx,.dbf,.prj,.cpg,.geojson,.json,.gpkg,.kml",
    multiple=True,
    description="📂 Upload AOI",
    button_style="primary"
)

layer_dropdown = widgets.Dropdown(
    options=["N/A"],
    value="N/A",
    description="GPKG Layer:",
    disabled=True,
    layout=widgets.Layout(width="500px"),
    style={"description_width": "120px"}
)

aoi_status_output = widgets.Output()

# ------------------------------------------------------------
# SCENE SELECTION
# ------------------------------------------------------------

scene_options = [(get_scene_label(row, idx), idx) for idx, row in df.iterrows()]

scene_dropdown = widgets.Dropdown(
    options=scene_options,
    description="Scene:",
    layout=widgets.Layout(width="100%"),
    style={"description_width": "120px"}
)

# ------------------------------------------------------------
# BAND MODE + BAND SELECTION
# ------------------------------------------------------------

band_mode = widgets.Dropdown(
    options=[("ALL AVAILABLE BANDS", "ALL"), ("SELECT BANDS", "SELECT")],
    value="ALL",
    description="Band Mode:",
    layout=widgets.Layout(width="500px"),
    style={"description_width": "120px"}
)

band_selection = widgets.SelectMultiple(
    options=[],
    value=(),
    description="Bands:",
    rows=10,
    layout=widgets.Layout(width="500px", height="230px"),
    style={"description_width": "120px"}
)

refresh_bands_button = widgets.Button(
    description="🔄 Load Available Bands",
    button_style="info",
    icon="refresh"
)

download_button = widgets.Button(
    description="⬇️ DOWNLOAD SCENE",
    button_style="success",
    icon="download",
    layout=widgets.Layout(width="300px", height="45px")
)

status_output = widgets.Output()


# ============================================================
# GLOBAL AOI STATE
# ============================================================

uploaded_gdf = None
uploaded_ee_aoi = None
current_vector_path = None
current_vector_kind = None


# ============================================================
# AOI LOAD + DISPLAY (shared by upload handler & layer dropdown)
# ============================================================

def load_and_report_aoi(path, kind, layer_name=None):
    global uploaded_gdf, uploaded_ee_aoi

    uploaded_gdf, uploaded_ee_aoi = load_vector_aoi(path, kind, layer_name)

    print()
    print("✅ AOI loaded successfully")
    print(f"   Features: {len(uploaded_gdf)}")
    print("   CRS: EPSG:4326")
    print("   AOI bounds:")
    print("   ", uploaded_gdf.total_bounds)

    scene_index = scene_dropdown.value
    row = df.loc[scene_index]
    lat, lon = get_query_coordinates(row)

    if lat is not None and lon is not None:
        query_point = ee.Geometry.Point([lon, lat])
        inside = uploaded_ee_aoi.contains(query_point).getInfo()

        print()
        print("📍 Query location:")
        print(f"   Latitude : {lat}")
        print(f"   Longitude: {lon}")

        if inside:
            print("   ✅ Query coordinate lies INSIDE uploaded AOI.")
        else:
            print("   ⚠️ WARNING: Query coordinate is OUTSIDE uploaded AOI.")
            if REQUIRE_POINT_INSIDE_AOI:
                print("   Download will be blocked until a valid AOI/scene combination is selected.")
    else:
        print()
        print("⚠️ Latitude/Longitude columns were not found in df.")


# ============================================================
# SHAPEFILE UPLOAD HANDLER
# ============================================================

def handle_aoi_upload(change):
    global current_vector_path, current_vector_kind

    if not upload_widget.value:
        return

    with aoi_status_output:
        clear_output()
        try:
            items = _extract_uploaded_items(upload_widget.value)

            print("📂 Uploaded file(s):")
            saved_paths = []
            for fname, content in items:
                print("   ", fname)
                fpath = os.path.join(WORKDIR, fname)
                with open(fpath, "wb") as f:
                    f.write(content)
                saved_paths.append(fpath)

            main_path = _resolve_main_vector_path(saved_paths)
            if main_path is None:
                raise ValueError(
                    "❌ Could not find a supported AOI file among uploads.\n"
                    "   Supported: .zip, .shp(+sidecars), .geojson/.json, .gpkg, .kml"
                )

            if main_path.lower().endswith(".zip"):
                actual_path, actual_kind = _extract_zip_and_find_vector(main_path)
            elif main_path.lower().endswith(".gpkg"):
                actual_path, actual_kind = main_path, "gpkg"
            elif main_path.lower().endswith((".geojson", ".json")):
                actual_path, actual_kind = main_path, "geojson"
            elif main_path.lower().endswith(".kml"):
                actual_path, actual_kind = main_path, "kml"
            elif main_path.lower().endswith(".shp"):
                actual_path, actual_kind = main_path, "shp"
            else:
                raise ValueError(f"❌ Unsupported AOI format: {main_path}")

            current_vector_path = actual_path
            current_vector_kind = actual_kind

            layers = _list_layers(actual_path) if actual_kind == "gpkg" else []

            if layers and len(layers) > 1:
                print()
                print(f"📚 GeoPackage has multiple layers: {layers}")
                print("   Pick the correct one from the 'GPKG Layer' dropdown below.")
                layer_dropdown.options = layers
                layer_dropdown.disabled = False
                layer_dropdown.value = layers[0]
                load_and_report_aoi(actual_path, actual_kind, layers[0])
            else:
                chosen_layer = layers[0] if layers else None
                layer_dropdown.options = [chosen_layer] if chosen_layer else ["N/A"]
                layer_dropdown.value = layer_dropdown.options[0]
                layer_dropdown.disabled = True
                load_and_report_aoi(actual_path, actual_kind, chosen_layer)

            load_bands()

        except Exception as e:
            print("❌ AOI upload failed:")
            print(str(e))


upload_widget.observe(handle_aoi_upload, names="value")


def handle_layer_change(change):
    if change["name"] != "value":
        return
    new_layer = change["new"]
    if not current_vector_path or new_layer in (None, "N/A"):
        return

    with aoi_status_output:
        clear_output()
        try:
            print(f"🔁 Switching to GeoPackage layer: '{new_layer}'")
            load_and_report_aoi(current_vector_path, current_vector_kind, new_layer)
        except Exception as e:
            print("❌ Could not load selected layer:")
            print(str(e))


layer_dropdown.observe(handle_layer_change, names="value")


# ============================================================
# LOAD AVAILABLE BANDS
# ============================================================

def load_bands(*args):
    with status_output:
        clear_output()
        try:
            scene_index = scene_dropdown.value
            row = df.loc[scene_index]
            asset_id = str(row["GEE_Asset_ID"])

            print("🛰️ Selected scene:")
            print("   ", asset_id)
            print()
            print("🔎 Reading available bands from Google Earth Engine...")

            img, available_bands = get_available_bands(asset_id)

            band_selection.options = available_bands
            band_selection.value = tuple(available_bands)

            print()
            print(f"✅ {len(available_bands)} bands available:")
            for i, band in enumerate(available_bands, start=1):
                print(f"   {i:02d}. {band}")

            print()
            print("Band mode:")
            print(f"   {band_mode.value}")

            if band_mode.value == "ALL":
                print("   ✅ ALL available bands will be downloaded.")
            else:
                print("   ☑ Select the required bands from the list.")

        except Exception as e:
            band_selection.options = []
            print("❌ Could not load bands:")
            print(str(e))


refresh_bands_button.on_click(load_bands)
scene_dropdown.observe(lambda change: load_bands() if change["name"] == "value" else None, names="value")


# ============================================================
# CHANGE BAND MODE
# ============================================================

def handle_band_mode(change):
    if change["name"] != "value":
        return
    mode = change["new"]
    if mode == "ALL":
        band_selection.value = tuple(band_selection.options)
        band_selection.disabled = True
    else:
        band_selection.disabled = False
        if not band_selection.value:
            band_selection.value = tuple(band_selection.options)


band_mode.observe(handle_band_mode, names="value")
band_selection.disabled = (band_mode.value == "ALL")


# ============================================================
# DOWNLOAD FUNCTION (with Google Drive fallback for oversized scenes)
# ============================================================

def download_selected_scene(button):
    global uploaded_gdf, uploaded_ee_aoi

    with status_output:
        clear_output()
        print("=" * 65)
        print("🛰️ SATELLITE SCENE DOWNLOAD")
        print("=" * 65)

        if CLIP_TO_SHAPEFILE and uploaded_ee_aoi is None:
            print()
            print("❌ Please upload an AOI file first.")
            return

        scene_index = scene_dropdown.value
        row = df.loc[scene_index]
        asset_id = str(row["GEE_Asset_ID"])

        print()
        print("Scene:")
        print("   ", asset_id)

        print()
        print("🔎 Loading scene from Google Earth Engine...")
        try:
            img = ee.Image(asset_id)
            available_bands = img.bandNames().getInfo()
        except Exception as e:
            print()
            print("❌ Could not load GEE image:")
            print(str(e))
            return

        print(f"   Available bands: {len(available_bands)}")

        if band_mode.value == "ALL":
            selected_bands = available_bands
            print()
            print("📡 BAND MODE: ALL AVAILABLE BANDS")
        else:
            selected_bands = list(band_selection.value)
            if len(selected_bands) == 0:
                print()
                print("❌ No bands selected.")
                return
            invalid = [b for b in selected_bands if b not in available_bands]
            if invalid:
                print()
                print("❌ Invalid bands:")
                print(invalid)
                return
            print()
            print("📡 BAND MODE: SELECTED BANDS")

        print()
        print(f"   Bands to download: {len(selected_bands)}")
        for b in selected_bands:
            print(f"      • {b}")

        img = img.select(selected_bands)

        # --------------------------------------------------------------
        # AOI validation + clip
        # --------------------------------------------------------------

        export_region_geom = None  # full ee.Geometry, used for the Drive fallback

        if CLIP_TO_SHAPEFILE:
            print()
            print("✂️ Checking query coordinate against uploaded AOI...")

            lat, lon = get_query_coordinates(row)

            if lat is not None and lon is not None:
                query_point = ee.Geometry.Point([lon, lat])
                try:
                    inside = uploaded_ee_aoi.contains(query_point).getInfo()
                except Exception as e:
                    print("❌ Could not check AOI:")
                    print(str(e))
                    return

                if not inside:
                    print()
                    if REQUIRE_POINT_INSIDE_AOI:
                        print("❌ DOWNLOAD BLOCKED")
                        print("   Query latitude/longitude is outside the uploaded AOI.")
                        print()
                        print(f"   Latitude : {lat}")
                        print(f"   Longitude: {lon}")
                        return
                    else:
                        print("⚠️ Query point is outside the AOI, but the block is disabled — continuing.")
                else:
                    print("   ✅ Query coordinate is inside AOI.")
            else:
                print("⚠️ Query coordinates not found.")
                print("   AOI clipping will still be used.")

            print()
            print("✂️ Clipping scene to uploaded AOI...")
            img = img.clip(uploaded_ee_aoi)
            export_region_geom = uploaded_ee_aoi
            region_coords = uploaded_ee_aoi.bounds().getInfo()["coordinates"]

        else:
            print()
            print("⚠️ AOI clipping disabled.")
            export_region_geom = img.geometry()
            region_coords = img.geometry().bounds().getInfo()["coordinates"]

        # --------------------------------------------------------------
        # Filename
        # --------------------------------------------------------------

        filename = OUTPUT_FILENAME.strip() or "AOI_Scene"
        if filename.lower().endswith((".tif", ".tiff", ".zip")):
            filename = os.path.splitext(filename)[0]

        # --------------------------------------------------------------
        # Try direct download; fall back to a Drive Export task if the
        # scene/AOI is too large for Earth Engine's direct-download limit.
        # --------------------------------------------------------------

        print()
        print("🔗 Attempting direct download from Earth Engine...")

        try:
            download_params = {
                "name": filename,
                "bands": selected_bands,
                "region": region_coords,
                "filePerBand": False,
                "format": "GEO_TIFF",
                "scale": EXPORT_SCALE_METERS,
            }
            download_url = img.getDownloadURL(download_params)

            response = requests.get(download_url, stream=True, timeout=600)
            response.raise_for_status()

            content_type = response.headers.get("content-type", "").lower()
            ext = ".zip" if ("zip" in content_type or len(selected_bands) > 1) else ".tif"
            output_path = os.path.join(OUTPUT_DIRECTORY, filename + ext)

            with open(output_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)

            file_size_mb = os.path.getsize(output_path) / (1024 * 1024)

            print()
            print("✅ DOWNLOAD COMPLETED (direct)")
            print()
            print("📁 File:")
            print("   ", output_path)
            print(f"   Size: {file_size_mb:.2f} MB")
            print()
            print(f"   Bands: {len(selected_bands)}")
            print(f"   AOI clipped: {CLIP_TO_SHAPEFILE}")

            if AUTO_DOWNLOAD:
                print()
                print("⬇️ Sending file to your computer...")
                try:
                    from google.colab import files
                    files.download(output_path)
                    print()
                    print("🎉 Done.")
                except Exception as e:
                    print(f"ℹ️ Could not trigger browser download automatically: {e}")

        except Exception as direct_err:

            print()
            print(f"⚠️ Direct download not possible: {direct_err}")

            if not ENABLE_DRIVE_FALLBACK:
                print("   ENABLE_DRIVE_FALLBACK is off, so the download stops here.")
                print("   Either shrink the AOI/bands/scale, or turn ENABLE_DRIVE_FALLBACK on")
                print("   at the top of this cell to auto-export large scenes to Drive.")
                return

            print()
            print("🔄 Scene is likely too large for a direct download (Earth Engine limits")
            print("   getDownloadURL requests). Submitting a GEE batch Export task to Google")
            print("   Drive instead — this runs on Earth Engine's servers, not this notebook.")

            safe_name = _safe_name(filename)

            try:
                task = ee.batch.Export.image.toDrive(
                    image=img,
                    description=safe_name[:100],
                    folder=DRIVE_EXPORT_FOLDER,
                    fileNamePrefix=safe_name,
                    region=export_region_geom,
                    scale=EXPORT_SCALE_METERS,
                    crs="EPSG:4326",
                    maxPixels=1e13,
                )
                task.start()

                print()
                print("🚀 Export task started.")
                print(f"   Task ID    : {task.id}")
                print(f"   Drive path : My Drive/{DRIVE_EXPORT_FOLDER}/{safe_name}.tif")
                print("   Track progress under the 'Tasks' tab at code.earthengine.google.com,")
                print("   or just check the Drive folder once the task finishes.")

            except Exception as export_err:
                print()
                print("❌ Could not start the Drive export task:")
                print(str(export_err))


download_button.on_click(download_selected_scene)


# ============================================================
# DISPLAY INTERFACE
# ============================================================

display(title)

display(widgets.HTML("<h3>1️⃣ Upload AOI File</h3>"))
display(upload_widget)
display(layer_dropdown)
display(aoi_status_output)

display(widgets.HTML("<h3>2️⃣ Select Scene from Current Query</h3>"))
display(scene_dropdown)

display(widgets.HTML("<h3>3️⃣ Band Selection</h3>"))
display(widgets.HBox([refresh_bands_button]))
display(widgets.VBox([band_mode, band_selection]))

display(widgets.HTML(
    "<h3>4️⃣ Download</h3>"
    "<p style='margin-top:-8px;color:#555;'>Clip / scale / output / Drive-fallback "
    "options are set in the fields at the top of this cell.</p>"
))
display(download_button)

display(widgets.HTML("<h3>📋 Status</h3>"))
display(status_output)

# Load bands initially for the first scene
load_bands()
